# Asymmetric training scenario: topology and initial load

This notebook visualizes the current `high_load_inner_asymmetric` scenario used for directional upper-PPO pretraining.

- Center gNB1 serves six movable eMBB UEs in the three-cell overlap.
- Left gNB0 contains three fixed eMBB UEs that create preload.
- Right gNB2 contains one fixed eMBB UE and therefore has more headroom.
- The desired upper strategy is to release center-cell traffic toward the right.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D

from global_ppo_3gnb_env import GlobalPPO3GNBEnv, SLICE_TYPES
from upper_agent_training_scenarios import CENTER_GAP_GNB_CONFIGS, get_upper_training_scenarios

GNB_NAMES = {0: 'Left gNB0', 1: 'Center gNB1', 2: 'Right gNB2'}
COLORS = {0: '#457b9d', 1: '#e76f51', 2: '#2a9d8f'}

In [ ]:
env = GlobalPPO3GNBEnv(
    seed=2,
    scenario_mode='curriculum',
    training_scenarios='high_load_inner_asymmetric',
    scenario_selection='cycle',
    gnb_configs=CENTER_GAP_GNB_CONFIGS['medium_270m'],
    upper_window_seconds=1.0,
    local_steps_per_global=10,
    radio_substeps=20,
    terminal_reward_only=False,
    safe_admission_enabled=True,
)
observation, info = env.reset(seed=2)
loads = np.asarray(info['load_matrix'], dtype=float)
counts = np.asarray(info['ue_count_matrix'], dtype=int)
ues = list(env.base_env.get_all_ues())

print('Scenario:', info['scenario_name'])
print('Observation shape:', observation.shape)
print('Episode duration:', info['episode_duration_s'], 's')

## Topology

Orange UEs are movable overlap UEs currently attached to the center. Gray UEs are fixed preload and are outside the three-cell overlap.

In [ ]:
configs = CENTER_GAP_GNB_CONFIGS['medium_270m']
fig, ax = plt.subplots(figsize=(14, 6))

for cfg in configs:
    gnb_id = int(cfg['id'])
    x, y = float(cfg['x']), float(cfg['y'])
    radius = float(cfg['coverage_radius'])
    ax.add_patch(Circle(
        (x, y), radius,
        fill=False, ls='--', lw=1.7,
        edgecolor=COLORS[gnb_id], alpha=.42,
    ))
    ax.scatter(x, y, marker='^', s=260, color=COLORS[gnb_id], zorder=5)
    ax.text(x, y + 50, GNB_NAMES[gnb_id], ha='center', weight='bold', fontsize=12)

for ue in ues:
    serving = int(ue.serving_gnb)
    movable = serving == 1 and abs(float(ue.x)) < 200.0
    color = '#f4a261' if movable else '#6c757d'
    marker = 'o' if movable else 's'
    ax.scatter(float(ue.x), float(ue.y), marker=marker, s=72, color=color, edgecolor='black', linewidth=.5, zorder=6)
    ax.text(float(ue.x), float(ue.y) + 16, f'UE{int(ue.id)}', ha='center', fontsize=8)

ax.annotate(
    'Desired offload direction',
    xy=(245, -105), xytext=(25, -105),
    arrowprops=dict(arrowstyle='->', lw=3, color='#2a9d8f'),
    color='#2a9d8f', weight='bold', ha='center',
)

legend = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f4a261', markeredgecolor='black', markersize=9, label='Movable overlap eMBB UE'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#6c757d', markeredgecolor='black', markersize=9, label='Fixed preload eMBB UE'),
]
ax.legend(handles=legend, loc='upper right')
ax.set(
    title='Three-gNB asymmetric directional-training topology',
    xlabel='x position (m)', ylabel='y position (m)',
    xlim=(-650, 650), ylim=(-180, 570),
)
ax.set_aspect('equal', adjustable='box')
ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

## Initial load and UE counts

In [ ]:
load_table = pd.DataFrame(
    loads,
    index=[GNB_NAMES[i] for i in range(3)],
    columns=SLICE_TYPES,
)
count_table = pd.DataFrame(
    counts,
    index=[GNB_NAMES[i] for i in range(3)],
    columns=SLICE_TYPES,
)

display(load_table.style.set_caption('Initial normalized load').format('{:.2f}'))
display(count_table.style.set_caption('Initial UE count'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
x = np.arange(3)

bars = axes[0].bar(x, loads[:, 0], color=[COLORS[i] for i in range(3)])
axes[0].axhline(loads[:, 0].mean(), color='#d62828', ls='--', lw=2, label=f'Balance target = {loads[:, 0].mean():.2f}')
axes[0].set_xticks(x, [GNB_NAMES[i] for i in range(3)])
axes[0].set_ylim(0, 1.0)
axes[0].set(title='Initial eMBB load per gNB', ylabel='normalized load')
axes[0].legend(); axes[0].grid(axis='y', alpha=.25)
for bar, value in zip(bars, loads[:, 0]):
    axes[0].text(bar.get_x()+bar.get_width()/2, value+.025, f'{value:.2f}', ha='center', weight='bold')

count_bars = axes[1].bar(x, counts[:, 0], color=[COLORS[i] for i in range(3)])
axes[1].set_xticks(x, [GNB_NAMES[i] for i in range(3)])
axes[1].set(title='Initial eMBB UE count per gNB', ylabel='UE count')
axes[1].grid(axis='y', alpha=.25)
for bar, value in zip(count_bars, counts[:, 0]):
    axes[1].text(bar.get_x()+bar.get_width()/2, value+.12, f'{int(value)}', ha='center', weight='bold')

plt.tight_layout(); plt.show()

In [ ]:
# Scenario consistency checks.
assert np.all((loads >= 0.0) & (loads <= 1.0))
assert np.all(loads.sum(axis=1) <= 1.0 + 1e-9)
assert loads[1, 0] > loads[0, 0] > loads[2, 0]
critical = env.critical_load_thresholds['eMBB']
assert loads[1, 0] >= critical, (
    f'Center eMBB utilization {loads[1, 0]:.3f} is below the '
    f'configured saturation threshold {critical:.3f}'
)
assert counts[:, 0].tolist() == [3, 6, 1]
movable = [ue for ue in ues if int(ue.serving_gnb) == 1]
assert len(movable) == 6
assert {abs(float(ue.x)) for ue in movable} == {132.0}
assert all(
    all(env.base_env._is_in_coverage(env.base_env._get_gnb_by_id(i), ue) for i in range(3))
    for ue in movable
)
print('PASS: topology, UE placement, coverage, load, and UE counts are coherent.')

## Interpretation

The initial eMBB load is:

```text
Left gNB0   = 0.51
Center gNB1 = 1.00
Right gNB2  = 0.12
```

These values are useful allocated PRBs divided by the physical 100-PRB gNB capacity. Center gNB1 is saturated, and right gNB2 is the preferred target because it has substantially more physical headroom than left gNB0.

# Complete scenario catalog

The following figures show all ten retained scenarios with the same medium `270 m` gNB topology. UE coordinates and slice composition change by scenario; the gNB geometry remains fixed.

In [ ]:
scenario_records = []
for scenario in get_upper_training_scenarios('all'):
    scenario_env = GlobalPPO3GNBEnv(
        seed=2,
        scenario_mode='curriculum',
        training_scenarios=scenario.name,
        scenario_selection='cycle',
        gnb_configs=CENTER_GAP_GNB_CONFIGS['medium_270m'],
        upper_window_seconds=1.0,
        local_steps_per_global=10,
        radio_substeps=20,
        terminal_reward_only=False,
        safe_admission_enabled=True,
    )
    try:
        _obs, scenario_info = scenario_env.reset(seed=2)
        scenario_ues = []
        for ue in scenario_env.base_env.get_all_ues():
            coverage = [
                gnb_id for gnb_id in range(3)
                if scenario_env.base_env._is_in_coverage(
                    scenario_env.base_env._get_gnb_by_id(gnb_id), ue
                )
            ]
            scenario_ues.append({
                'id': int(ue.id), 'x': float(ue.x), 'y': float(ue.y),
                'serving': int(ue.serving_gnb), 'slice': str(ue.slice_type),
                'coverage': coverage,
            })
        scenario_records.append({
            'name': scenario.name,
            'description': scenario.description,
            'duration_s': scenario.duration_s,
            'loads': np.asarray(scenario_info['load_matrix'], dtype=float),
            'counts': np.asarray(scenario_info['ue_count_matrix'], dtype=int),
            'ues': scenario_ues,
        })
    finally:
        scenario_env.close()

catalog_summary = pd.DataFrame([
    {
        'scenario': record['name'],
        'duration (s)': record['duration_s'],
        'UEs': len(record['ues']),
        'gNB0 total load': record['loads'][0].sum(),
        'gNB1 total load': record['loads'][1].sum(),
        'gNB2 total load': record['loads'][2].sum(),
    }
    for record in scenario_records
])
display(catalog_summary.style.format({
    'duration (s)': '{:.0f}', 'gNB0 total load': '{:.2f}',
    'gNB1 total load': '{:.2f}', 'gNB2 total load': '{:.2f}',
}))

## All scenario topologies

Circles are overlap UEs covered by multiple gNBs. Squares are fixed/core UEs covered only by their source. Colors identify slices.

In [ ]:
SLICE_COLORS = {'eMBB': '#f4a261', 'URLLC': '#e63946', 'mMTC': '#6a4c93'}
fig, axes = plt.subplots(5, 2, figsize=(16, 25))
axes = axes.reshape(-1)
for ax, record in zip(axes, scenario_records):
    for cfg in CENTER_GAP_GNB_CONFIGS['medium_270m']:
        gnb_id = int(cfg['id'])
        ax.add_patch(Circle(
            (cfg['x'], cfg['y']), cfg['coverage_radius'],
            fill=False, ls='--', lw=1.0,
            edgecolor=COLORS[gnb_id], alpha=.28,
        ))
        ax.scatter(cfg['x'], cfg['y'], marker='^', s=105, color=COLORS[gnb_id], zorder=4)
        ax.text(cfg['x'], cfg['y'] + 42, f'gNB{gnb_id}', ha='center', weight='bold', fontsize=8)
    for ue in record['ues']:
        overlap = len(ue['coverage']) == 3
        ax.scatter(
            ue['x'], ue['y'], marker='o' if overlap else 's', s=34,
            color=SLICE_COLORS[ue['slice']], edgecolor='black', linewidth=.35, zorder=5,
        )
    total = record['loads'].sum(axis=1)
    ax.set_title(
        f"{record['name']}\ninitial gNB totals: {total[0]:.2f}, {total[1]:.2f}, {total[2]:.2f}",
        fontsize=10,
    )
    ax.set(xlim=(-650, 650), ylim=(-180, 570), xlabel='x (m)', ylabel='y (m)')
    ax.set_aspect('equal', adjustable='box'); ax.grid(alpha=.15)

topology_legend = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=SLICE_COLORS[s], markeredgecolor='black', markersize=8, label=s)
    for s in SLICE_TYPES
] + [
    Line2D([0], [0], marker='o', color='black', lw=0, markersize=7, label='3-cell overlap'),
    Line2D([0], [0], marker='s', color='black', lw=0, markersize=7, label='fixed/core'),
]
fig.legend(handles=topology_legend, loc='upper center', ncol=5, bbox_to_anchor=(.5, .995))
fig.suptitle('Complete slice-aware scenario topology catalog', fontsize=16, y=1.01)
fig.tight_layout(rect=(0, 0, 1, .985)); plt.show()

## Initial load of every scenario

Each subplot shows the per-slice normalized load at each gNB.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(16, 22), sharey=False)
axes = axes.reshape(-1)
x = np.arange(3)
width = .24
for ax, record in zip(axes, scenario_records):
    for s_idx, slice_name in enumerate(SLICE_TYPES):
        bars = ax.bar(
            x + (s_idx - 1) * width,
            record['loads'][:, s_idx], width,
            label=slice_name, color=SLICE_COLORS[slice_name],
        )
        for bar, value in zip(bars, record['loads'][:, s_idx]):
            if value > 0:
                ax.text(bar.get_x()+bar.get_width()/2, value+.025, f'{value:.2f}', ha='center', fontsize=7)
    ax.set_xticks(x, ['gNB0', 'gNB1', 'gNB2'])
    ax.set_ylim(0, max(1.05, float(record['loads'].max()) + .15))
    ax.set(title=record['name'], ylabel='normalized slice load')
    ax.grid(axis='y', alpha=.2)
    ax.legend(fontsize=8)
fig.suptitle('Initial per-gNB, per-slice load for all scenarios', fontsize=16, y=1.01)
fig.tight_layout(); plt.show()

In [ ]:
assert len(scenario_records) == 10
assert {record['name'] for record in scenario_records} == {
    scenario.name for scenario in get_upper_training_scenarios('all')
}
assert all(record['loads'].shape == (3, 3) for record in scenario_records)
assert all(record['counts'].shape == (3, 3) for record in scenario_records)
assert all(np.all(np.isfinite(record['loads'])) for record in scenario_records)
print('PASS: all 10 scenario topologies and initial load matrices were generated.')

## Handover and load-distribution feasibility audit

The oracle below is not a trained policy. It releases from above-average sources only toward below-average, radio-feasible targets. With allocated utilization, handovers are physically possible, but utilization is not conserved: queues and scheduler decisions can make source and targets saturate independently. The table therefore reports which scenarios improve and which need redesign for allocated-utilization training.

In [ ]:
def oracle_action(env, loads):
    means = loads.mean(axis=0)
    action = np.zeros((3, 2, 3), dtype=np.float32)
    for source in range(3):
        for s_idx in range(3):
            if loads[source, s_idx] <= means[s_idx] + 1e-9:
                continue
            deficits = [max(means[s_idx] - loads[target, s_idx], 0.0) for target in env.neighbors[source]]
            total = sum(deficits)
            if total <= 0:
                continue
            for slot, target in enumerate(env.neighbors[source]):
                feasible = any(
                    int(ue.serving_gnb) == source
                    and env.slice_types.index(str(ue.slice_type)) == s_idx
                    and env.base_env._is_in_coverage(env.base_env._get_gnb_by_id(target), ue)
                    and env.base_env._measure_rsrp(env.base_env._get_gnb_by_id(target), ue)
                        - env.base_env._measure_rsrp(env.base_env._get_gnb_by_id(source), ue) > -5.0
                    for ue in env.base_env.get_all_ues()
                )
                if feasible and deficits[slot] > 0:
                    action[source, slot, s_idx] = -deficits[slot] / total
    return action

feasibility_rows = []
for scenario in get_upper_training_scenarios('all'):
    audit_env = GlobalPPO3GNBEnv(
        seed=2, scenario_mode='curriculum', training_scenarios=scenario.name,
        scenario_selection='cycle', gnb_configs=CENTER_GAP_GNB_CONFIGS['medium_270m'],
        upper_window_seconds=1.0, local_steps_per_global=10, radio_substeps=20,
        terminal_reward_only=False, safe_admission_enabled=True,
    )
    try:
        _obs, reset_info = audit_env.reset(seed=2)
        start = np.asarray(reset_info['load_matrix'], dtype=float)
        action = oracle_action(audit_env, start)
        _obs, reward, _terminated, _truncated, result = audit_env.step(action.reshape(-1))
        end = np.asarray(result['load_matrix_end'], dtype=float)
        physically_bounded = bool(np.all(start.sum(axis=1) <= 1.0 + 1e-9))
        feasibility_rows.append({
            'scenario': scenario.name,
            'initial imbalance': result['load_imbalance_start'],
            'final imbalance': result['load_imbalance_end'],
            'improvement': result['load_imbalance_start'] - result['load_imbalance_end'],
            'handovers': result['handover_count'],
            'reward': reward,
            'initial allocation bounded': physically_bounded,
            'handover possible': bool(result['handover_count'] > 0),
            'balance improved': bool(result['load_imbalance_end'] < result['load_imbalance_start']),
        })
    finally:
        audit_env.close()

feasibility = pd.DataFrame(feasibility_rows)
display(feasibility.style.format({
    'initial imbalance': '{:.3f}', 'final imbalance': '{:.3f}',
    'improvement': '{:.3f}', 'reward': '{:.3f}',
}))
assert feasibility['initial allocation bounded'].all()
assert feasibility['handover possible'].all()
print('PASS: all scenarios are physically bounded initially and can generate handovers.')
print('Scenarios with no allocated-utilization improvement need redesign before training:')
display(feasibility.loc[~feasibility['balance improved'], ['scenario', 'initial imbalance', 'final imbalance', 'handovers']])

In [ ]:
env.close()